In [1]:
import pandas as pd

inflation_df = pd.read_excel('/content/datasets2/inflation.xlsx')

display(inflation_df.head())

,Socio économique\n\nTunisie; Unité,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 126,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135
0,,,Jan 2015,Fév 2015,Mar 2015,Avr 2015,Mai 2015,Jui 2015,Jul 2015,Aoû 2015,...,Mai 2025,Jui 2025,Jul 2025,Aoû 2025,Sep 2025,Oct 2025,Nov 2025,Déc 2025,Jan 2026,Fév 2026
1,Glissement annuel de l'Indice de Prix à la Con...,(2015 = 100),5,5.3,5.2,5.2,4.8,4.5,3.7,3.8,...,5.4,5.4,5.3,5.2,5,4.9,4.9,4.9,4.8,5
2,Produits alimentaires et boissons non alcoolisées,(2015 = 100),5.9,6.6,6.9,6.2,5.5,4.3,3.2,3,...,6.7,6.4,5.9,5.9,5.7,5.6,5.8,6.1,5.9,6.7
3,Produits alimentaires,(2015 = 100),5.9,6.5,6.8,6.1,5.3,4.1,2.9,2.8,...,7,6.6,6.1,6.1,5.9,5.8,5.9,6.4,6.1,7
4,Pain et céréales,(2015 = 100),3.1,3.1,3.1,2.4,2.8,2.9,2.6,2.6,...,3.3,3.2,2.9,2.8,2.8,2.8,2.7,2.9,2.8,3


In [2]:
inflation_df = inflation_df.iloc[:2]
display(inflation_df.head())

,Socio économique\n\nTunisie; Unité,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 126,Unnamed: 127,Unnamed: 128,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135
0,,,Jan 2015,Fév 2015,Mar 2015,Avr 2015,Mai 2015,Jui 2015,Jul 2015,Aoû 2015,...,Mai 2025,Jui 2025,Jul 2025,Aoû 2025,Sep 2025,Oct 2025,Nov 2025,Déc 2025,Jan 2026,Fév 2026
1,Glissement annuel de l'Indice de Prix à la Con...,(2015 = 100),5,5.3,5.2,5.2,4.8,4.5,3.7,3.8,...,5.4,5.4,5.3,5.2,5,4.9,4.9,4.9,4.8,5


In [3]:
import pandas as pd

df = inflation_df.iloc[:2]

# FIX: start from column 2, not 3
dates = df.iloc[0, 2:].values
inflation = df.iloc[1, 2:].values

clean_df = pd.DataFrame({
    "Date": dates,
    "Inflation": inflation
})

display(clean_df.head())
display(clean_df.tail())

,Date,Inflation
0,Jan 2015,5
1,Fév 2015,5.3
2,Mar 2015,5.2
3,Avr 2015,5.2
4,Mai 2015,4.8


,Date,Inflation
129,Oct 2025,4.9
130,Nov 2025,4.9
131,Déc 2025,4.9
132,Jan 2026,4.8
133,Fév 2026,5


In [4]:
import pandas as pd

# prendre seulement les 2 lignes utiles
df = inflation_df.iloc[:2]

dates = df.iloc[0, 2:].values
inflation = df.iloc[1, 2:].values


month_map = {
    "Jan": "Jan",
    "Fév": "Feb",
    "Fev": "Feb",
    "Mar": "Mar",
    "Avr": "Apr",
    "Mai": "May",
    "Jui": "Jun",
    "Jul": "Jul",
    "Aoû": "Aug",
    "Aou": "Aug",
    "Sep": "Sep",
    "Oct": "Oct",
    "Nov": "Nov",
    "Déc": "Dec",
    "Dec": "Dec"
}

clean_dates = []
for d in dates:
    for fr, en in month_map.items():
        d = str(d).replace(fr, en)
    clean_dates.append(d)


clean_df = pd.DataFrame({
    "Date": pd.to_datetime(clean_dates, format="%b %Y"),
    "Inflation": inflation
})

clean_df = clean_df.set_index("Date").sort_index()
clean_df = clean_df.iloc[:-1]
display(clean_df.head())
display(clean_df.tail())

,Inflation
Date,
2015-01-01,5
2015-02-01,5.3
2015-03-01,5.2
2015-04-01,5.2
2015-05-01,4.8


,Inflation
Date,
2025-09-01,5
2025-10-01,4.9
2025-11-01,4.9
2025-12-01,4.9
2026-01-01,4.8


we distribute monthly inflation evenly across days using exponential compounding
When this is valid

✔ forecasting models
✔ ML feature engineering
✔ simulations
✔ smoothing time series
Monthly truth
   ↓
continuous exponential smoothing
   ↓
fake daily curve
Yes, the logic is correct mathematically.

But conceptually:

you didn’t discover daily inflation
you invented a smooth curve that matches monthly inflation

In [6]:
import numpy as np
import pandas as pd

monthly = clean_df.copy()

# 1. Convert properly BEFORE index
monthly.index = pd.to_datetime(monthly.index)

# 2. Clean inflation
monthly["Inflation"] = pd.to_numeric(monthly["Inflation"], errors="coerce")

# 3. Drop missing
monthly = monthly.dropna(subset=["Inflation"])

# 4. Feature engineering
monthly["factor"] = 1 + monthly["Inflation"] / 100
monthly["daily_factor"] = monthly["factor"] ** (1/30) - 1

# 5. Resample works because index is datetime
daily = monthly.resample("D").ffill()

daily["Inflation_daily"] = daily["daily_factor"] * 100

daily = daily[['Inflation_daily']]

display(daily.head())
display(daily.tail())

,Inflation_daily
Date,
2015-01-01,0.162766
2015-01-02,0.162766
2015-01-03,0.162766
2015-01-04,0.162766
2015-01-05,0.162766


,Inflation_daily
Date,
2025-12-28,0.159585
2025-12-29,0.159585
2025-12-30,0.159585
2025-12-31,0.159585
2026-01-01,0.156401


In [7]:
daily.reset_index().to_excel("inflation_daily.xlsx", index=False)